# 🎬 Final Phase — Notebook 4: End-to-End Demo
## Quantum-Enhanced License Plate Recognition — Clean Showcase

---
This is the **presentation-ready notebook** — everything runs in order, with no setup required beyond mounting Drive and installing dependencies.

**What it demonstrates:**
1. Both models loaded and ready
2. Night-time plate recognition challenge visualized
3. Side-by-side Quantum vs Classical predictions
4. Final performance summary

---
### 🌙 The Night-Time LPR Challenge
Standard LPR systems fail under low-light conditions. This system uses:
- **ZeroDCE** to recover dark images
- **Quantum computing** to navigate the complex feature space
- **LSTM + CTC** to decode the full plate as text


## Step 1 — Install & Mount

In [ ]:
print('🔧 Installing dependencies...')
!pip install pennylane pennylane-lightning editdistance -q

import os, json, time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
import pandas as pd
import pennylane as qml
import editdistance
from torch.utils.data import Dataset, random_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from google.colab import drive
drive.mount('/content/drive')

print(f'\n✅ Ready!  Device: {device}')
print(f'   PennyLane: {qml.__version__}')
print(f'   PyTorch:   {torch.__version__}')

## Step 2 — Configure Paths

In [ ]:
# ─────────────────────────────────────────────────────────────
# 🔧 UPDATE THESE TO MATCH YOUR GOOGLE DRIVE:
PROJECT_PATH   = '/content/drive/MyDrive/Quantum_LPR_Project'
CHECKPOINT_DIR = os.path.join(PROJECT_PATH, 'checkpoints')
CSV_PATH       = '/content/drive/MyDrive/License Plate/Manifests/2_train_hr_images.csv'
ZIP_PATH       = '/content/drive/MyDrive/License Plate/wYe7pBJ7-train.zip'
EXTRACT_PATH   = '/content/lpr_train'
# ─────────────────────────────────────────────────────────────

CHARS    = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
IDX2CHAR = {i+1: c for i, c in enumerate(CHARS)}
N_QUBITS = 8
SEED     = 42

import zipfile
if not os.path.exists(os.path.join(EXTRACT_PATH, 'train')):
    print('📂 Extracting dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)

print('✅ Paths configured.')

## Step 3 — Define Models

In [ ]:
class ZeroDCE_Light(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,16,3,1,1), nn.ReLU(),
            nn.Conv2d(16,16,3,1,1), nn.ReLU(),
            nn.Conv2d(16,24,3,1,1), nn.Tanh())
    def forward(self, x):
        A, e = self.conv(x), x
        for i in range(8):
            e = e + A[:,3*i:3*(i+1)] * (torch.pow(e,2) - e)
        return e

dev_qml = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev_qml, interface='torch')
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_layer = qml.qnn.TorchLayer(quantum_circuit,
                                           {'weights': (2, N_QUBITS, 3)})
    def forward(self, x): return self.q_layer(x)

class HybridLPRNet_8Q(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        self.enhancer  = ZeroDCE_Light()
        self.features  = nn.Sequential(
            nn.Conv2d(3,64,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(64,128,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(128,N_QUBITS,1,1))
        self.quantum   = QuantumLayer()
        self.rnn       = nn.LSTM(N_QUBITS,128,bidirectional=True,batch_first=True)
        self.classifier= nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.enhancer(x)
        x = self.features(x)
        b,c,h,w = x.size()
        x_seq = x.mean(dim=2).permute(0,2,1)
        q_out = self.quantum(x_seq.reshape(-1,N_QUBITS)).reshape(b,w,N_QUBITS)
        return self.classifier(self.rnn(q_out)[0]).permute(1,0,2)

class ClassicalLPRNet(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        self.enhancer  = ZeroDCE_Light()
        self.features  = nn.Sequential(
            nn.Conv2d(3,64,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(64,128,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(128,N_QUBITS,1,1))
        self.classical = nn.Sequential(nn.Linear(N_QUBITS,16),nn.Tanh(),
                                        nn.Linear(16,N_QUBITS))
        self.rnn       = nn.LSTM(N_QUBITS,128,bidirectional=True,batch_first=True)
        self.classifier= nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.enhancer(x)
        x = self.features(x)
        b,c,h,w = x.size()
        x = self.classical(x.mean(dim=2).permute(0,2,1))
        return self.classifier(self.rnn(x)[0]).permute(1,0,2)

print('✅ Model classes defined.')

## Step 4 — Load Trained Checkpoints

In [ ]:
def load_model(model, best_file, fallback_file, model_name):
    for fname in [best_file, fallback_file]:
        path = os.path.join(CHECKPOINT_DIR, fname)
        if os.path.exists(path):
            ckpt = torch.load(path, map_location=device)
            model.load_state_dict(ckpt['model_state_dict'])
            ep  = ckpt.get('epoch', '?')
            cer = ckpt.get('val_cer', '?')
            print(f'  ✅  {model_name}: loaded {fname}  (epoch {ep}, best_CER={cer})')
            return True
    # Fallback: if totally no checkpoint, log warning
    print(f'  ⚠️   {model_name}: no checkpoint found — using random weights!')
    return False

q_model = HybridLPRNet_8Q(37).to(device)
c_model = ClassicalLPRNet(37).to(device)

print('Loading models...')
q_ok = load_model(q_model, '8qubit_best.pth',    '8qubit_model.pth',   'Quantum ⚡')
c_ok = load_model(c_model, 'classical_best.pth', 'classical_model.pth','Classical 🔷')

q_model.eval(); c_model.eval()
print()

# Parameter count
q_params = sum(p.numel() for p in q_model.parameters() if p.requires_grad)
c_params = sum(p.numel() for p in c_model.parameters() if p.requires_grad)
print(f'  ⚡ Quantum  model parameters: {q_params:,}')
print(f'  🔷 Classical model parameters: {c_params:,}')
print(f'  📈 Quantum overhead: +{q_params-c_params:,} params ({(q_params/c_params-1)*100:.1f}%)')

## Step 5 — Dataset & CTC Decoder

In [ ]:
class LPRDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, night=False):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir   = root_dir
        self.transform  = transform
        self.night      = night
    def __len__(self): return len(self.data_frame)
    def make_night(self, img):
        g = np.random.uniform(2.0, 3.5)
        return torch.clamp(torch.pow(img,g) + torch.randn_like(img)*0.05, 0, 1)
    def __getitem__(self, idx):
        row   = self.data_frame.iloc[idx]
        path  = row['path']
        label = str(row['label']).upper()
        full  = path if path.startswith('/content') else os.path.join(self.root_dir, path)
        try:   img = Image.open(full).convert('RGB')
        except: img = Image.new('RGB', (256,64))
        if self.transform: img = self.transform(img)
        if self.night: img = self.make_night(img)
        return img, label

transform   = transforms.Compose([transforms.Resize((64,256)), transforms.ToTensor()])

torch.manual_seed(SEED)
clean_ds    = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform, night=False)
night_ds    = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform, night=True)

N               = len(clean_ds)
n_train, n_val  = int(N*0.70), int(N*0.15)
n_test          = N - n_train - n_val
_, _, test_sub  = random_split(range(N), [n_train, n_val, n_test])
TEST_INDICES    = list(test_sub)

def ctc_greedy_decode(log_probs):
    indices = torch.argmax(log_probs, dim=2)
    batch   = []
    for b in range(indices.size(1)):
        seq, chars, prev = indices[:,b].tolist(), [], -1
        for i in seq:
            if i != 0 and i != prev: chars.append(IDX2CHAR.get(i,''))
            prev = i
        batch.append(''.join(chars))
    return batch

print(f'✅ Test samples available: {n_test}')

## Step 6 — 🌙 Night-Time Recognition Demo (10 samples)

In [ ]:
N_DEMO = 10
np.random.seed(SEED)
demo_indices = np.random.choice(TEST_INDICES, N_DEMO, replace=False)

# ── Figure layout: N rows × 5 columns ────────────────────
fig = plt.figure(figsize=(24, 3.8 * N_DEMO))
fig.patch.set_facecolor('#111111')

col_headers = [
    'Clean Original', 'Night Input 🌙', 'ZeroDCE Enhanced',
    'Quantum ⚡ Result', 'Classical 🔷 Result'
]

outer = fig.add_gridspec(N_DEMO + 1, 5, hspace=0.08, wspace=0.05,
                          top=0.97, bottom=0.02, left=0.01, right=0.99)

# Header row
for col, header in enumerate(col_headers):
    ax = fig.add_subplot(outer[0, col])
    ax.set_facecolor('#222222')
    ax.text(0.5, 0.5, header, ha='center', va='center',
            color='white', fontsize=11, fontweight='bold')
    ax.axis('off')

q_correct = c_correct = 0

for row, idx in enumerate(demo_indices):
    clean_img, label = clean_ds[int(idx)]
    night_img, _     = night_ds[int(idx)]

    inp = night_img.unsqueeze(0).to(device)
    with torch.no_grad():
        enhanced = q_model.enhancer(inp).cpu().squeeze(0)
        q_pred   = ctc_greedy_decode(q_model(inp).cpu())[0]
        c_pred   = ctc_greedy_decode(c_model(inp).cpu())[0]

    q_ok = (q_pred == label)
    c_ok = (c_pred == label)
    if q_ok: q_correct += 1
    if c_ok: c_correct += 1

    def img_ax(col_idx):
        ax = fig.add_subplot(outer[row+1, col_idx])
        ax.axis('off')
        return ax

    def show_img(ax, tensor, border_color='#555555'):
        arr = np.clip(tensor.permute(1,2,0).numpy(), 0, 1)
        ax.imshow(arr, aspect='auto')
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2.5)
        ax.tick_params(left=False, bottom=False,
                       labelleft=False, labelbottom=False)

    # Col 1: Clean
    ax0 = img_ax(0)
    show_img(ax0, clean_img)
    ax0.set_title(f'True: {label}', color='#AAAAAA', fontsize=9, pad=3)

    # Col 2: Night
    ax1 = img_ax(1)
    show_img(ax1, night_img, '#3949AB')

    # Col 3: Enhanced
    ax2 = img_ax(2)
    show_img(ax2, enhanced, '#00897B')

    # Col 4: Quantum pred
    ax3 = img_ax(3)
    ax3.set_facecolor('#1B5E20' if q_ok else '#B71C1C')
    ax3.text(0.5, 0.5,
             f"{'✅' if q_ok else '❌'}  {q_pred if q_pred else '(blank)'}\n"
             f"(True: {label})",
             ha='center', va='center', color='white',
             fontsize=12, fontweight='bold')

    # Col 5: Classical pred
    ax4 = img_ax(4)
    ax4.set_facecolor('#1B5E20' if c_ok else '#B71C1C')
    ax4.text(0.5, 0.5,
             f"{'✅' if c_ok else '❌'}  {c_pred if c_pred else '(blank)'}\n"
             f"(True: {label})",
             ha='center', va='center', color='white',
             fontsize=12, fontweight='bold')

demo_path = os.path.join(PROJECT_PATH, 'demo_result.png')
plt.savefig(demo_path, dpi=120, bbox_inches='tight', facecolor='#111111')
plt.show()

print(f'\n✅ Demo complete — saved demo_result.png')
print(f'\n   On this {N_DEMO}-sample demo:')
print(f'   ⚡ Quantum  correct: {q_correct}/{N_DEMO}  ({q_correct*10}%)')
print(f'   🔷 Classical correct: {c_correct}/{N_DEMO}  ({c_correct*10}%)')

## Step 7 — Final Performance Summary

In [ ]:
# Load saved comparison CSV if available, else show demo results
csv_path = os.path.join(PROJECT_PATH, 'final_comparison_table.csv')

if os.path.exists(csv_path):
    table = pd.read_csv(csv_path)
    print('\n' + '='*65)
    print('  FINAL RESULTS — QUANTUM vs CLASSICAL LPR')
    print('='*65)
    print(table.to_string(index=False))
    print('='*65)
else:
    print('\n⚠️  Full evaluation table not found.')
    print('    Run Notebook 02_Evaluation_Suite.ipynb first for full metrics.')
    print()
    print(f'    Demo results ({N_DEMO} samples):')
    print(f'    ⚡ Quantum  Accuracy: {q_correct*10}%')
    print(f'    🔷 Classical Accuracy: {c_correct*10}%')

print()
print('🏁 Demo notebook complete!')
print('   ▶ To see full evaluation: run 02_Evaluation_Suite.ipynb')
print('   ▶ To see interpretability: run 03_Visualizations.ipynb')

---
## 📖 Project Summary

### Architecture: HybridLPRNet_8Q
```
Input Image (Night/Dark)
       ↓
  ZeroDCE Enhancer      → Recovers details from dark images
       ↓
  CNN Feature Extractor → Extracts 8 spatial feature channels
       ↓
  8-Qubit Quantum Layer → AngleEmbedding + StronglyEntanglingLayers
       ↓                   operates in 256-dimensional Hilbert space
  Bi-LSTM Decoder       → Reads quantum features as a sequence
       ↓
  CTC Loss + Decode     → Outputs full plate text (e.g., 'MH12AB1234')
```

### Why Quantum?
- The 8-qubit circuit operates in `2^8 = 256` dimensional space
- `StronglyEntanglingLayers` creates non-local correlations across all 8 feature channels simultaneously
- This helps distinguish **visually similar characters** (`0` vs `O`, `1` vs `I`, `5` vs `S`) under noise — a task classical linear layers struggle with
- Classical replacement (FC layer) only operates in 8-dimensional space — far less expressive
